Slaps on coverpage and scratchwork page. Automatically removes the answerkey page

In [3]:
from pdfrw import PdfReader, PdfWriter
import os.path
testname='2018Su_Test3v'
joinfilepath='C:/Users/souri/Dropbox/MiscTeachingStuff/pre_n_post/'


for n in range(5):
    inpfn=testname+str(n)+'.pdf'
    writer = PdfWriter()
    mcp=joinfilepath+'MidtermCoverPage'+str(n)+'.pdf'
    sp=joinfilepath+'scratchwork.pdf'
    if os.path.exists(inpfn) and os.path.exists(mcp) and os.path.exists(sp):
        writer.addpages(PdfReader(mcp).pages)
        writer.addpages(PdfReader(inpfn).pages[:-1])
        writer.addpages(PdfReader(sp).pages)
        opfn='Test_v'+str(n)+'.pdf'
        print("Writing "+opfn)
        writer.write(opfn)
    

Writing Test_v0.pdf
Writing Test_v1.pdf
Writing Test_v2.pdf
Writing Test_v3.pdf


The following script checks the accuracy of serial numbers in All.xlsx using the serial numbers given in BSGrades. It does a left and right merge. 


Nan on the right means they didn't write the correct srl no, and on the left means unknown serial number found

Removes the rows which are OK, i.e. the name entered for that serial number matches the person that serial number was assigned to (the earlier merging process returns some OK rows)

In [1]:
import pandas as pd
def find_oks(row):
    return row['Name'].find(row['First Name'].upper())!=-1
def remove_ok_rows(df):
    notnull=df.dropna()
    okrows=notnull.loc[notnull.apply(find_oks, axis=1)]
    oksremoved=pd.concat([df, okrows]).drop_duplicates(keep=False)
    return oksremoved

df=pd.read_csv('BSGrades.csv', usecols=["Last Name", "First Name", "Serial Number Text Grade <Text>"])
df_all=pd.read_excel('All.xlsx', header=None, parse_cols=1,names = ["Serial Number Text Grade <Text>", "Name"])

checker_left = pd.merge(df,df_all[['Serial Number Text Grade <Text>','Name']],on='Serial Number Text Grade <Text>', how='left')
checker_right = pd.merge(df,df_all[['Serial Number Text Grade <Text>','Name']],on='Serial Number Text Grade <Text>', how='right')
u1=checker_left[checker_left.isnull().any(axis=1)]
u2=checker_left[checker_left['Serial Number Text Grade <Text>'].duplicated(keep=False)]
u3=checker_right[checker_right.isnull().any(axis=1)]
error_rpt=pd.concat([u1, u2, u3], axis=0)
if not error_rpt.empty:
    error_rpt=remove_ok_rows(error_rpt)
error_rpt

,Last Name,First Name,Serial Number Text Grade <Text>,Name
1,Bolling,Atiyaa,1003,NaN
13,Obi,Simon,1012,NaN
9,Kim,Lucas,1011,KIM DONG M
10,Kim,Lucas,1011,OBI SIMON


The following script transfers MP grades and test grades to the (downloaded) BS Gradebook

Steps:

Download MP as MP.csv. 

Calculate the scores in MP and put them in a column titled "MP Score"

Make sure All_processed.csv is ready (All_processed is the output of Grader)

Make sure Brightspace gradebook is downloaded as BSGrades.csv, and "out of" for tests are correct. 

The following code will create 3 columns extra columns in BSGrades for MPScores, test scores and test missed.

Then it will copy them over to the correct columns. Pay attention to the TWO #### below. 

Correct 'Test # Questions Missed Text Grade ' and 

Correct 'Midterm # Points Grade Numeric MaxPoints:40 Weight:33.333333333 Category:Midterms CategoryWeight:60>'
 

In [2]:
import pandas as pd

In [5]:
df=pd.read_csv('BSGrades.csv')

df_mp=pd.read_csv('MP.csv')
df_mp=df_mp.rename(columns={'Student ID': 'Serial Number Text Grade <Text>'})

df = pd.merge(df,df_mp[['Serial Number Text Grade <Text>','MP Score']],on='Serial Number Text Grade <Text>', how='left')

df_all=pd.read_csv('All_processed.csv')
df_all=df_all.rename(columns={'Srl No': 'Serial Number Text Grade <Text>'})

df = pd.merge(df,df_all[['Serial Number Text Grade <Text>','Score']],on='Serial Number Text Grade <Text>', how='left')
df = pd.merge(df,df_all[['Serial Number Text Grade <Text>','Missed']],on='Serial Number Text Grade <Text>', how='left')

df['Mastering Physics Points Grade <Numeric MaxPoints:100 Weight:14.75>']=df['MP Score']

##### CHECK THIS CHECK THIS MAKE SURE YOU ARE DOING THE CORRECT TEST
df['Midterm 3 Points Grade <Numeric MaxPoints:40 Weight:33.333333333 Category:Midterms CategoryWeight:60>']=df['Score']

##### CHECK THIS CHECK THIS MAKE SURE YOU ARE DOING THE CORRECT TEST
df['Test 3 Questions Missed Text Grade <Text>']=df['Missed']

columns = ['MP Score', 'Score', 'Missed']
df.drop(columns, inplace=True, axis=1)

df.to_csv('BSGrades.csv', index=False)

In [6]:
error_rpt.empty

True